# macro_extreme_risk_retrain_v1

Executable diagnostic and training-gate notebook for a narrow, lagged macro extreme-risk feature pack.

Governance: baseline runtime composer is `v19reg_v22_budget_usage_bull_guard_v1`; direct actor baseline is `balanced_rule_residual_v22`; direct failed macro baseline is `balanced_macro_v1`. The changed layer is `feature`, and the candidate profile is `macro_extreme_risk_retrain_v1`.

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model" / "card" / "macro_extreme_risk_retrain_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.settings import CSV_PATH
from src.soxl_quant.data import CORE_FEATURE_COLUMNS, engineer_feature_frame, feature_columns_for_packs

SUMMARY_PATH = CARD / "summary.json"
print("root=.")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model/card/macro_extreme_risk_retrain_v1


In [2]:
panel = pd.read_csv(ROOT / CSV_PATH, parse_dates=["date"]).sort_values("date").set_index("date")
panel["soxl_ret1"] = panel["soxl_close"].pct_change()
panel["soxx_ret1"] = panel["soxx_close"].pct_change()
panel["abs_ret1"] = panel["soxl_ret1"].abs()
panel["gap"] = panel["soxl_open"] / panel["soxl_close"].shift(1) - 1.0
panel["close_from_open"] = panel["soxl_close"] / panel["soxl_open"] - 1.0
panel["intraday_range_pct"] = panel["soxl_high"] / panel["soxl_low"] - 1.0
panel["vix_chg_5"] = panel["vix"].pct_change(5)
panel["volume_z_21"] = (panel["soxl_volume"] - panel["soxl_volume"].rolling(21).mean()) / (panel["soxl_volume"].rolling(21).std() + 1e-8)
panel["rv21"] = panel["soxl_ret1"].rolling(21).std() * np.sqrt(252)
panel["dd63"] = panel["soxl_close"] / panel["soxl_close"].rolling(63).max() - 1.0

extreme = panel.loc[panel["abs_ret1"] >= 0.15].copy()
losses = panel.nsmallest(15, "soxl_ret1")
gains = panel.nlargest(15, "soxl_ret1")
quantiles = (panel["soxl_ret1"].quantile([0.001, 0.005, 0.01, 0.025, 0.05, 0.95, 0.975, 0.99, 0.995, 0.999]) * 100).round(2)

extreme_summary = {
    "rows": int(panel["soxl_ret1"].notna().sum()),
    "start": str(panel.index.min().date()),
    "end": str(panel.index.max().date()),
    "abs_ge_15_days": int(len(extreme)),
    "loss_days_le_minus_15": int((extreme["soxl_ret1"] <= -0.15).sum()),
    "gain_days_ge_15": int((extreme["soxl_ret1"] >= 0.15).sum()),
    "worst_day": str(panel["soxl_ret1"].idxmin().date()),
    "worst_return_pct": round(float(panel["soxl_ret1"].min() * 100), 2),
    "best_day": str(panel["soxl_ret1"].idxmax().date()),
    "best_return_pct": round(float(panel["soxl_ret1"].max() * 100), 2),
}
extreme_by_year = (
    extreme.assign(year=extreme.index.year)
    .groupby("year")
    .agg(days=("soxl_ret1", "size"), worst=("soxl_ret1", "min"), best=("soxl_ret1", "max"), avg_vix=("vix", "mean"))
)
for col in ["worst", "best"]:
    extreme_by_year[col] = extreme_by_year[col] * 100

display(pd.DataFrame([extreme_summary]))
display(quantiles.to_frame("return_pct"))
display(extreme_by_year.round(2).tail(12))

show_cols = ["soxl_ret1", "soxl_open", "soxl_high", "soxl_low", "soxl_close", "soxl_volume", "gap", "close_from_open", "intraday_range_pct", "vix", "soxx_ret1"]
display((losses[show_cols].assign(soxl_ret1=lambda x: x["soxl_ret1"]*100, soxx_ret1=lambda x: x["soxx_ret1"]*100, gap=lambda x: x["gap"]*100, close_from_open=lambda x: x["close_from_open"]*100, intraday_range_pct=lambda x: x["intraday_range_pct"]*100)).round(2))
display((gains[show_cols].assign(soxl_ret1=lambda x: x["soxl_ret1"]*100, soxx_ret1=lambda x: x["soxx_ret1"]*100, gap=lambda x: x["gap"]*100, close_from_open=lambda x: x["close_from_open"]*100, intraday_range_pct=lambda x: x["intraday_range_pct"]*100)).round(2))

,rows,start,end,abs_ge_15_days,loss_days_le_minus_15,gain_days_ge_15,worst_day,worst_return_pct,best_day,best_return_pct
0,4083,2010-03-11,2026-06-04,78,39,39,2020-03-16,-38.59,2025-04-09,54.79


,return_pct
0.001,-25.27
0.005,-17.30
0.010,-14.83
0.025,-11.82
0.050,-8.68
0.950,9.11
0.975,11.63
0.990,14.87
0.995,17.10
0.999,26.84


,days,worst,best,avg_vix
year,,,,
2011,5,-17.30,17.10,37.28
2014,1,-20.72,-20.72,21.24
2016,1,-16.88,-16.88,25.76
2018,2,-19.78,17.19,27.82
2019,2,-17.77,17.10,22.17
2020,16,-38.59,34.11,55.76
2021,4,-17.21,17.91,28.90
2022,15,-18.30,30.74,28.21
2023,4,15.25,19.48,18.69


,soxl_ret1,soxl_open,soxl_high,soxl_low,soxl_close,soxl_volume,gap,close_from_open,intraday_range_pct,vix,soxx_ret1
date,,,,,,,,,,,
2020-03-16,-38.59,5.99,7.10,5.27,5.32,47341500,-30.90,-11.14,34.76,82.69,-15.23
2020-03-12,-30.41,7.77,8.92,6.80,6.80,47734500,-20.48,-12.49,31.08,75.47,-10.96
2025-04-03,-29.83,13.81,14.12,11.40,11.41,302517400,-15.07,-17.38,23.86,30.02,-10.09
2020-03-18,-25.89,4.60,5.26,3.57,4.33,67953000,-21.32,-5.81,47.59,76.45,-8.22
2020-03-09,-25.34,10.14,11.79,9.87,9.89,31191000,-23.45,-2.46,19.52,54.46,-8.54
2025-04-10,-24.59,10.92,11.14,8.35,9.63,686331800,-14.49,-11.81,33.41,40.72,-8.17
2025-04-04,-23.49,10.05,10.43,8.18,8.73,499158100,-11.92,-13.13,27.51,45.31,-7.47
2025-01-27,-23.25,27.66,28.08,23.84,25.06,198431800,-15.28,-9.40,17.79,17.90,-7.84
2024-09-03,-22.48,36.80,36.81,29.40,30.07,140077900,-5.13,-18.29,25.20,20.72,-7.63


,soxl_ret1,soxl_open,soxl_high,soxl_low,soxl_close,soxl_volume,gap,close_from_open,intraday_range_pct,vix,soxx_ret1
date,,,,,,,,,,,
2025-04-09,54.79,8.48,13.11,8.24,12.77,787748800,2.79,50.59,59.10,33.62,18.57
2020-03-24,34.11,5.57,6.27,5.55,6.25,67669500,19.53,12.20,12.96,61.67,10.82
2022-11-10,30.74,10.80,12.17,10.56,12.08,196270300,16.88,11.85,15.25,23.53,10.35
2020-04-06,30.68,6.13,7.18,6.13,7.10,51997500,12.84,15.80,17.19,45.24,10.20
2020-03-13,27.32,8.07,8.73,6.53,8.66,43620000,18.69,7.27,33.67,57.83,10.73
2025-05-12,21.40,16.89,17.47,16.50,17.13,263806000,19.70,1.42,5.88,18.39,7.18
2020-03-26,20.00,6.47,7.27,6.27,7.21,55215000,7.74,11.38,15.96,61.00,6.72
2024-08-08,19.98,28.57,31.60,26.87,31.46,149988600,8.96,10.12,17.60,23.79,6.90
2023-05-26,19.48,20.05,23.26,20.03,22.88,112964600,4.70,14.11,16.13,17.95,6.54


In [3]:
def make_qlib_like_frames(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    soxl = pd.DataFrame(
        {
            "$close": df["soxl_close"],
            "$open": df["soxl_open"],
            "$high": df["soxl_high"],
            "$low": df["soxl_low"],
            "$volume": df["soxl_volume"],
            "$vix": df["vix"],
            "Mean($close, 5)": df["soxl_close"].rolling(5).mean(),
            "Mean($close, 21)": df["soxl_close"].rolling(21).mean(),
            "Mean($close, 63)": df["soxl_close"].rolling(63).mean(),
            "Mean($close, 180)": df["soxl_close"].rolling(180).mean(),
            "Std($close / Ref($close, 1) - 1, 10)": df["soxl_close"].pct_change().rolling(10).std(),
            "Std($close / Ref($close, 1) - 1, 21)": df["soxl_close"].pct_change().rolling(21).std(),
            "Std($close / Ref($close, 1) - 1, 63)": df["soxl_close"].pct_change().rolling(63).std(),
            "Mean($volume, 21)": df["soxl_volume"].rolling(21).mean(),
        }
    )
    soxx = pd.DataFrame(
        {
            "$close": df["soxx_close"],
            "$open": df["soxx_open"],
            "$high": df["soxx_high"],
            "$low": df["soxx_low"],
            "$volume": df["soxx_volume"],
        }
    )
    return soxl, soxx

soxl, soxx = make_qlib_like_frames(panel)
feature_df, price = engineer_feature_frame(soxl, soxx, feature_packs=("macro_extreme_risk_v1",))
flat = feature_df.droplevel("instrument")
price_flat = price.droplevel("instrument")
pack_columns = [col for col in feature_columns_for_packs(("macro_extreme_risk_v1",)) if col not in CORE_FEATURE_COLUMNS]
print(f"feature rows={len(flat)} columns={len(flat.columns)} pack_columns={len(pack_columns)}")
display(pd.DataFrame({"pack_column": pack_columns}))
display(flat[pack_columns].describe().T[["mean", "std", "min", "max"]].round(4))

feature rows=3812 columns=70 pack_columns=11


,pack_column
0,MACRO_EXTREME_VIX_TERM_SLOPE_LAG1
1,MACRO_EXTREME_VIX_SHORT_RATIO_LAG1
2,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1
3,MACRO_EXTREME_SKEW_ZSCORE_252_LAG1
4,MACRO_EXTREME_YC_10Y2Y_ZSCORE_252_LAG1
5,MACRO_EXTREME_YC_10Y2Y_DELTA_21_LAG1
6,MACRO_EXTREME_DXY_ZSCORE_252_LAG1
7,MACRO_EXTREME_DXY_ROC_63_LAG1
8,MACRO_EXTREME_OIL_ROC_63_LAG1
9,MACRO_EXTREME_RISK_PRESSURE_LAG1


,mean,std,min,max
MACRO_EXTREME_VIX_TERM_SLOPE_LAG1,-0.1063,0.0757,-0.2896,0.3437
MACRO_EXTREME_VIX_SHORT_RATIO_LAG1,-0.0531,0.0983,-0.3407,0.5900
MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,0.0216,1.1589,-2.4788,8.3119
MACRO_EXTREME_SKEW_ZSCORE_252_LAG1,0.1388,1.1709,-3.3599,4.9746
MACRO_EXTREME_YC_10Y2Y_ZSCORE_252_LAG1,-0.1684,1.4608,-3.7209,5.2082
MACRO_EXTREME_YC_10Y2Y_DELTA_21_LAG1,-0.0117,0.1533,-0.7600,0.6100
MACRO_EXTREME_DXY_ZSCORE_252_LAG1,0.5030,1.3033,-2.4052,6.4223
MACRO_EXTREME_DXY_ROC_63_LAG1,0.0052,0.0272,-0.0711,0.0911
MACRO_EXTREME_OIL_ROC_63_LAG1,0.0225,0.2334,-2.1041,3.6869
MACRO_EXTREME_RISK_PRESSURE_LAG1,0.4259,0.2765,0.0155,2.2383


In [4]:
def rolling_ic(feature: pd.Series, target: pd.Series, window: int = 252) -> pd.Series:
    return feature.rolling(window).corr(target).replace([np.inf, -np.inf], np.nan)

future_returns = {h: price_flat.shift(-h) / price_flat - 1.0 for h in [1, 5, 10, 20]}
ic_rows = []
for feature_name in pack_columns:
    x = flat[feature_name].astype(float)
    for horizon, y in future_returns.items():
        aligned = pd.concat([x, y.rename("target")], axis=1).dropna()
        if len(aligned) < 300 or aligned[feature_name].std() == 0:
            continue
        overall_ic = float(aligned[feature_name].corr(aligned["target"]))
        ric = rolling_ic(aligned[feature_name], aligned["target"]).dropna()
        ic_mean = float(ric.mean()) if len(ric) else np.nan
        ic_std = float(ric.std(ddof=1)) if len(ric) > 1 else np.nan
        ic_t = float(ic_mean / (ic_std / math.sqrt(len(ric)))) if ic_std and not np.isnan(ic_std) and ic_std > 0 else np.nan
        ic_rows.append({
            "feature": feature_name,
            "horizon": horizon,
            "overall_ic": overall_ic,
            "rolling_ic_mean": ic_mean,
            "rolling_ic_positive_ratio": float((ric > 0).mean()) if len(ric) else np.nan,
            "rolling_ic_t": ic_t,
            "n": int(len(aligned)),
        })

ic_df = pd.DataFrame(ic_rows).sort_values(["rolling_ic_mean", "overall_ic"], ascending=False)
display(ic_df.round(4).head(30))

corr_rows = []
for feature_name in pack_columns:
    corr = flat[[*CORE_FEATURE_COLUMNS, feature_name]].corr(numeric_only=True)[feature_name].drop(feature_name).abs().dropna()
    top = corr.sort_values(ascending=False).head(3)
    corr_rows.append({
        "feature": feature_name,
        "max_abs_core_corr": float(top.iloc[0]) if len(top) else np.nan,
        "top_core_corr": top.index[0] if len(top) else None,
        "top3_core_corr": "; ".join(f"{idx}:{val:.3f}" for idx, val in top.items()),
    })
corr_df = pd.DataFrame(corr_rows).sort_values("max_abs_core_corr", ascending=True)
display(corr_df.round(4))

,feature,horizon,overall_ic,rolling_ic_mean,rolling_ic_positive_ratio,rolling_ic_t,n
3,MACRO_EXTREME_VIX_TERM_SLOPE_LAG1,20,0.0994,0.1805,0.8297,62.3424,3792
2,MACRO_EXTREME_VIX_TERM_SLOPE_LAG1,10,0.0556,0.1297,0.8054,49.5790,3802
39,MACRO_EXTREME_RISK_PRESSURE_LAG1,20,-0.0076,0.1016,0.7365,35.5946,3792
38,MACRO_EXTREME_RISK_PRESSURE_LAG1,10,-0.0043,0.0962,0.7299,41.2951,3802
7,MACRO_EXTREME_VIX_SHORT_RATIO_LAG1,20,0.0218,0.0957,0.7933,39.2264,3792
10,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,10,0.0455,0.0942,0.6745,35.6591,3802
1,MACRO_EXTREME_VIX_TERM_SLOPE_LAG1,5,0.0320,0.0941,0.8088,46.3545,3807
11,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,20,0.0476,0.0876,0.6202,29.5687,3792
37,MACRO_EXTREME_RISK_PRESSURE_LAG1,5,-0.0021,0.0715,0.6963,40.4046,3807
9,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,5,0.0304,0.0696,0.6814,33.8762,3807


,feature,max_abs_core_corr,top_core_corr,top3_core_corr
5,MACRO_EXTREME_YC_10Y2Y_DELTA_21_LAG1,0.1812,rolling_new_low_ratio_63,rolling_new_low_ratio_63:0.181; below_ma63_day...
10,MACRO_EXTREME_RISK_PRESSURE_RELIEF_5_LAG1,0.2963,trend_gap_21_velocity,trend_gap_21_velocity:0.296; vol_coef_21:0.281...
4,MACRO_EXTREME_YC_10Y2Y_ZSCORE_252_LAG1,0.3538,decay_63,decay_63:0.354; anchor_slope_180:0.281; tracki...
8,MACRO_EXTREME_OIL_ROC_63_LAG1,0.3668,trend_slope_63,trend_slope_63:0.367; mom_63:0.353; rolling_ne...
3,MACRO_EXTREME_SKEW_ZSCORE_252_LAG1,0.3747,regime_risk,regime_risk:0.375; drawdown_63:0.366; regime_r...
7,MACRO_EXTREME_DXY_ROC_63_LAG1,0.3903,mom_63,mom_63:0.390; regime_risk:0.380; trend_gap_63:...
6,MACRO_EXTREME_DXY_ZSCORE_252_LAG1,0.4166,anchor_gap_180,anchor_gap_180:0.417; mom_63:0.385; vix_percen...
1,MACRO_EXTREME_VIX_SHORT_RATIO_LAG1,0.5283,vix_zscore,vix_zscore:0.528; vix_percentile_252:0.457; cu...
2,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,0.6463,vix_zscore,vix_zscore:0.646; vix_percentile_252:0.542; vi...
9,MACRO_EXTREME_RISK_PRESSURE_LAG1,0.7603,vix_zscore,vix_zscore:0.760; vix_percentile_252:0.673; re...


In [5]:
def signal_metrics(signal: pd.Series, forward_return: pd.Series) -> dict[str, float]:
    aligned = pd.concat([signal.rename("signal"), forward_return.rename("ret")], axis=1).dropna()
    active = aligned[aligned["signal"] > 0]
    if active.empty:
        return {"signals": 0, "mean_ret": np.nan, "accuracy": np.nan, "profit_factor": np.nan, "sharpe": np.nan}
    rets = active["ret"]
    gains = rets[rets > 0].sum()
    losses = -rets[rets < 0].sum()
    return {
        "signals": int(len(active)),
        "mean_ret": float(rets.mean()),
        "accuracy": float((rets > 0).mean()),
        "profit_factor": float(gains / losses) if losses > 0 else np.inf,
        "sharpe": float(rets.mean() / (rets.std(ddof=1) + 1e-8) * math.sqrt(252 / 5)),
    }

backtest_rows = []
forward_5 = future_returns[5]
for feature_name in pack_columns:
    x = flat[feature_name].astype(float)
    lo = x.quantile(0.20)
    hi = x.quantile(0.80)
    candidates = {
        "top_quintile_long": (x >= hi).astype(float),
        "bottom_quintile_long": (x <= lo).astype(float),
    }
    for name, signal in candidates.items():
        metrics = signal_metrics(signal, forward_5)
        metrics.update({"feature": feature_name, "rule": name})
        backtest_rows.append(metrics)
backtest_df = pd.DataFrame(backtest_rows).sort_values(["profit_factor", "accuracy"], ascending=False)
display(backtest_df.round(4).head(30))

,signals,mean_ret,accuracy,profit_factor,sharpe,feature,rule
8,763,0.0270,0.5950,1.7552,1.4687,MACRO_EXTREME_YC_10Y2Y_ZSCORE_252_LAG1,top_quintile_long
19,758,0.0222,0.5765,1.7100,1.4537,MACRO_EXTREME_RISK_PRESSURE_LAG1,bottom_quintile_long
3,763,0.0198,0.5898,1.6855,1.3918,MACRO_EXTREME_VIX_SHORT_RATIO_LAG1,bottom_quintile_long
13,758,0.0198,0.5726,1.5173,1.1453,MACRO_EXTREME_DXY_ZSCORE_252_LAG1,bottom_quintile_long
0,763,0.0251,0.5806,1.5117,1.1276,MACRO_EXTREME_VIX_TERM_SLOPE_LAG1,top_quintile_long
4,763,0.0214,0.5845,1.4726,1.0416,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,top_quintile_long
14,763,0.0191,0.5701,1.4608,1.0367,MACRO_EXTREME_DXY_ROC_63_LAG1,top_quintile_long
12,763,0.0191,0.5701,1.4544,1.0115,MACRO_EXTREME_DXY_ZSCORE_252_LAG1,top_quintile_long
15,763,0.0145,0.5478,1.4381,0.9849,MACRO_EXTREME_DXY_ROC_63_LAG1,bottom_quintile_long
5,763,0.0140,0.5662,1.4225,0.9174,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,bottom_quintile_long


In [6]:
def read_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

macro_summary = read_json(ROOT / "runtime/model/card/balanced_macro_v1/soxl_balanced_macro_v1_summary.json")
v22_summary = read_json(ROOT / "runtime/model/card/balanced_rule_residual_v22/soxl_balanced_rule_residual_v22_summary.json")
composer_review = read_json(ROOT / "runtime/model/card/v19reg_v22_budget_usage_bull_guard_v1/runtime_promotion_review.json")

def fixed_split_rows(summary: dict, label: str) -> list[dict]:
    rows = []
    for split, metrics in (summary.get("slices") or {}).items():
        rows.append({"profile": label, "split": split, **{k: metrics.get(k) for k in ["return", "sharpe", "max_dd", "avg_pos", "relative_return"]}})
    if summary.get("test_proxy"):
        metrics = summary["test_proxy"]
        rows.append({"profile": label, "split": "test_proxy", **{k: metrics.get(k) for k in ["return", "sharpe", "max_dd", "avg_pos", "relative_return"]}})
    return rows

comparison_df = pd.DataFrame([
    *fixed_split_rows(v22_summary, "balanced_rule_residual_v22"),
    *fixed_split_rows(macro_summary, "balanced_macro_v1"),
])
if not comparison_df.empty:
    display(comparison_df.round(4))

composer_windows = []
for window, payload in (composer_review.get("windows") or {}).items():
    candidate = payload.get("candidate", {})
    composer_windows.append({"profile": "v19reg_v22_budget_usage_bull_guard_v1", "window": window, **{k: candidate.get(k) for k in ["return", "sharpe", "max_dd", "avg_pos", "relative_return"]}})
if composer_windows:
    display(pd.DataFrame(composer_windows).round(4))

,profile,split,return,sharpe,max_dd,avg_pos,relative_return
0,balanced_rule_residual_v22,test_2020,115.6025,2.7429,16.2229,0.4149,None
1,balanced_rule_residual_v22,test_2022,-24.5226,-0.9114,32.7112,0.2695,None
2,balanced_rule_residual_v22,test_2023_2024,37.8516,0.6603,32.4257,0.4468,None
3,balanced_rule_residual_v22,test_proxy,695.4516,0.9412,54.5180,0.5085,None


,profile,window,return,sharpe,max_dd,avg_pos,relative_return
0,v19reg_v22_budget_usage_bull_guard_v1,wf_2020,98.2709,1.4560,26.1393,0.3989,40.1773
1,v19reg_v22_budget_usage_bull_guard_v1,wf_2022,-20.6833,-0.6242,26.6510,0.1835,65.9048
2,v19reg_v22_budget_usage_bull_guard_v1,wf_2023_2024,58.1947,1.3264,30.1832,0.3368,58.5514


In [7]:
gate_rows = []
for feature_name in pack_columns:
    feature_ic = ic_df[ic_df["feature"] == feature_name]
    corr = corr_df[corr_df["feature"] == feature_name].iloc[0]
    best_ic = feature_ic.sort_values("rolling_ic_mean", ascending=False).iloc[0]
    best_bt = backtest_df[backtest_df["feature"] == feature_name].sort_values("profit_factor", ascending=False).iloc[0]
    gate_rows.append({
        "feature": feature_name,
        "best_horizon": int(best_ic["horizon"]),
        "best_rolling_ic_mean": float(best_ic["rolling_ic_mean"]),
        "best_rolling_ic_positive_ratio": float(best_ic["rolling_ic_positive_ratio"]),
        "best_ic_t": float(best_ic["rolling_ic_t"]),
        "max_abs_core_corr": float(corr["max_abs_core_corr"]),
        "best_pf": float(best_bt["profit_factor"]),
        "best_accuracy": float(best_bt["accuracy"]),
        "signals": int(best_bt["signals"]),
        "full_gate": bool(
            best_ic["rolling_ic_mean"] > 0.03
            and best_ic["rolling_ic_positive_ratio"] > 0.55
            and best_ic["rolling_ic_t"] > 2.0
            and corr["max_abs_core_corr"] <= 0.80
            and best_bt["profit_factor"] > 1.2
            and best_bt["accuracy"] > 0.55
        ),
    })
gate_df = pd.DataFrame(gate_rows).sort_values(["full_gate", "best_rolling_ic_mean", "best_pf"], ascending=False)
display(gate_df.round(4))

passing_features = gate_df.loc[gate_df["full_gate"], "feature"].tolist()
train_gate_decision = "ready_for_smoke_retrain" if passing_features else "do_not_train_gate_failed"
if train_gate_decision == "ready_for_smoke_retrain" and len(passing_features) < 2:
    train_gate_decision = "diagnostic_only_single_feature_pass"

summary = {
    "profile": "macro_extreme_risk_retrain_v1",
    "feature_pack": "macro_extreme_risk_v1",
    "generated_at": pd.Timestamp.utcnow().isoformat(),
    "extreme_summary": extreme_summary,
    "feature_count": len(feature_columns_for_packs(("vix_term", "macro_extreme_risk_v1"))),
    "pack_columns": pack_columns,
    "gate_decision": train_gate_decision,
    "passing_features": passing_features,
    "best_gate_rows": gate_df.head(8).to_dict(orient="records"),
    "balanced_macro_v1_promotion": macro_summary.get("promotion", {}),
    "runtime_composer_decision": composer_review.get("decision"),
    "artifacts": {
        "notebook": str((CARD / "macro_extreme_risk_retrain_v1.ipynb").relative_to(ROOT)),
        "summary_json": str(SUMMARY_PATH.relative_to(ROOT)),
        "profile": "runtime/model/card/macro_extreme_risk_retrain_v1/profile.yaml",
    },
}
smoke_summary_path = CARD / "soxl_macro_extreme_risk_retrain_v1_smoke2_summary.json"
if smoke_summary_path.exists():
    smoke_summary = read_json(smoke_summary_path)
    fixed_checks = ((smoke_summary.get("promotion") or {}).get("fixed_split") or {}).get("checks", {})
    failed_checks = {key: value for key, value in fixed_checks.items() if not value.get("passed")}
    summary["gate_decision"] = "feature_gate_passed_smoke_chain_passed_not_promoted"
    summary["smoke_result"] = {
        "status": "passed",
        "artifact_tag": "smoke2",
        "seed": smoke_summary.get("seed"),
        "max_iters": 2,
        "episode_per_iter": 2,
        "episode_len": 128,
        "summary_json": str(smoke_summary_path.relative_to(ROOT)),
        "actor_retention": "not_committed_registry_keep_actor_false",
        "valid": smoke_summary.get("valid", {}),
        "test_proxy": smoke_summary.get("test_proxy", {}),
        "test_2022": (smoke_summary.get("slices") or {}).get("test_2022", {}),
        "promotion": smoke_summary.get("promotion", {}),
    }
    summary["promotion_decision"] = {
        "eligible": False,
        "reason": "smoke chain passed, but fixed-split promotion checks fail on 2022 max drawdown and average position; full retrain should add execution or risk-budget control before promotion consideration.",
        "failed_checks": failed_checks,
    }
    summary["next_action"] = "Use macro_extreme_risk_v1 as a candidate feature pack only behind a risk-budget/execution guard; do not promote from smoke evidence."
    summary["updated_at"] = pd.Timestamp.utcnow().isoformat()
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps({"gate_decision": summary["gate_decision"], "passing_features": passing_features, "summary": str(SUMMARY_PATH.relative_to(ROOT))}, indent=2))


,feature,best_horizon,best_rolling_ic_mean,best_rolling_ic_positive_ratio,best_ic_t,max_abs_core_corr,best_pf,best_accuracy,signals,full_gate
0,MACRO_EXTREME_VIX_TERM_SLOPE_LAG1,20,0.1805,0.8297,62.3424,0.7692,1.5117,0.5806,763,True
9,MACRO_EXTREME_RISK_PRESSURE_LAG1,20,0.1016,0.7365,35.5946,0.7603,1.7100,0.5765,758,True
1,MACRO_EXTREME_VIX_SHORT_RATIO_LAG1,20,0.0957,0.7933,39.2264,0.5283,1.6855,0.5898,763,True
2,MACRO_EXTREME_VVIX_ZSCORE_252_LAG1,10,0.0942,0.6745,35.6591,0.6463,1.4726,0.5845,763,True
7,MACRO_EXTREME_DXY_ROC_63_LAG1,10,0.0459,0.6117,17.6751,0.3903,1.4608,0.5701,763,True
10,MACRO_EXTREME_RISK_PRESSURE_RELIEF_5_LAG1,5,0.0327,0.6482,25.1735,0.2963,1.3235,0.5611,1800,True
6,MACRO_EXTREME_DXY_ZSCORE_252_LAG1,10,0.0283,0.5714,10.2667,0.4166,1.5173,0.5726,758,False
4,MACRO_EXTREME_YC_10Y2Y_ZSCORE_252_LAG1,20,0.0238,0.5177,5.2920,0.3538,1.7552,0.5950,763,False
5,MACRO_EXTREME_YC_10Y2Y_DELTA_21_LAG1,20,0.0199,0.5126,6.4805,0.1812,1.3945,0.5623,770,False
3,MACRO_EXTREME_SKEW_ZSCORE_252_LAG1,1,-0.0205,0.4022,-25.4468,0.3747,1.3291,0.5400,763,False


{
  "gate_decision": "feature_gate_passed_smoke_chain_passed_not_promoted",
  "passing_features": [
    "MACRO_EXTREME_VIX_TERM_SLOPE_LAG1",
    "MACRO_EXTREME_RISK_PRESSURE_LAG1",
    "MACRO_EXTREME_VIX_SHORT_RATIO_LAG1",
    "MACRO_EXTREME_VVIX_ZSCORE_252_LAG1",
    "MACRO_EXTREME_DXY_ROC_63_LAG1",
    "MACRO_EXTREME_RISK_PRESSURE_RELIEF_5_LAG1"
  ],
  "summary": "runtime/model/card/macro_extreme_risk_retrain_v1/summary.json"
}


## Conclusion

This notebook is the primary conclusion surface. The machine-readable decision is written to `summary.json`; the smoke training bundle is recorded in `runtime/model/card/macro_extreme_risk_retrain_v1/soxl_macro_extreme_risk_retrain_v1_smoke2_summary.json`.

Decision: `macro_extreme_risk_v1` passed the feature-screen gate and the training/evaluation artifact chain passed a 2-iteration smoke run, but the candidate is not promotion eligible. Smoke metrics: test proxy return 2337.92%, Sharpe 1.14, MaxDD 60.68%; 2022 return -55.67%, Sharpe -1.45, MaxDD 59.22%, avg position 0.393.

Promotion rule outcome: fixed-split checks failed on 2022 MaxDD (59.22% > 55.00%) and 2022 avg position (0.393 > 0.350). The next experiment should keep the narrow macro pack but add explicit risk-budget or execution gating before any full retrain or promotion claim.
